# 01. 데이터 전처리

> 팀 공동 전처리 코드다. 처리 기준은 팀 전체 논의로 정했고 팀원 1명이 작성했다. RQ2 결과를 재현하기 위해 이 저장소에 함께 포함했다.

조달청 벤처나라 통합실적 원본(52,582행 × 43열)을 8단계로 정리해 분석용 단일 CSV(52,210행 × 23열)를 만든다.

| 단계 | 처리 | 결과 |
|---|---|---|
| 1 | 수량·단가 보정 (단가 ≤ 5, 또는 수량 ≥ 1만 & 단가 ≤ 10이면 교환) | 90행 |
| 2 | 업체명 정제 (`주식회사`·`(주)`·영문 괄호 제거) | 2,278 → 2,263개 |
| 3 | 실적금액 0원 행 삭제 | 372행 |
| 4 | 불필요 열 삭제 (U열, W~AO열) | 43 → 23열 |
| 5 | 실적일자 → datetime | |
| 6 | 벤처업체번호 `CN` 제거 → int | |
| 7 | 수요기관코드·품명 Label Encoding | |
| 8 | 완전 중복 행 제거 | 0행 |

> 원본 CSV는 저장소에 포함하지 않는다. `data/README.md`를 참고해 `data/raw/`에 넣은 뒤 실행한다.

In [1]:
import re
import pandas as pd

SRC = "../data/raw/조달청_벤처나라_통합실적_20261001_물품분류번호추가 (2조 최종).csv"
DST = "../data/조달청_벤처나라_전처리결과.csv"

df = pd.read_csv(SRC, encoding="utf-8-sig", low_memory=False)
print("원본:", df.shape)

원본: (52582, 43)


## 4. 불필요 열 삭제
U열(세부물품분류번호)과 W~AO열(품목 상세·상태 열)을 지우고 V열(품명)은 남긴다.

In [2]:
cols = list(df.columns)
drop_cols = [cols[20]] + cols[22:41]
df = df.drop(columns=drop_cols)
print("삭제한 열:", drop_cols)

삭제한 열: ['세부물품분류번호', '영문품명', '세부품명', '세부영문품명', '한글품목명', '제조업체명', '상품모델명', '품목등록일시', '품목변경일시', '품목입력일자', '제품인증목록', '품목구분', '구성품여부', '품목삭제여부', '품목사용여부', '물품이미지URL', '품목정보출처', '개별품목명매핑상태', '물품분류번호검증상태', '품목정보충돌여부']


## 1. 수량·단가 보정
실적금액 = 수량 × 단가 관계를 기준으로, 두 값이 뒤바뀐 것으로 보이는 행을 교환한다.

In [3]:
qty, price = df["판매물품수량"], df["물품단가"]
swap = (
    ((price <= 5) & (qty > price))          # 단가가 0~5로 너무 작고 수량이 더 큰 경우
    | ((qty >= 10000) & (price <= 10))      # 수량 1만 이상인데 단가가 0~10인 경우
)
df.loc[swap, ["판매물품수량", "물품단가"]] = df.loc[swap, ["물품단가", "판매물품수량"]].values
df[["판매물품수량", "물품단가"]] = df[["판매물품수량", "물품단가"]].astype("int64")
print("수량/단가 교환 행 수:", int(swap.sum()))

수량/단가 교환 행 수: 90


## 2. 업체명 정제

In [4]:
def clean_name(name: str) -> str:
    name = str(name).strip()
    # 'Name(한글명)' 형태 -> 한글명만 (예: 3J Trading(3제이 트레이딩) -> 3제이 트레이딩)
    m = re.fullmatch(r"[^가-힣()]*[A-Za-z][^가-힣()]*\s*\((.*[가-힣].*)\)", name)
    if m:
        name = m.group(1)
    # (주), 주식회사 삭제
    name = re.sub(r"\(\s*주\s*\)|주식회사", "", name)
    # 한글명 뒤에 붙은 영문 괄호 문구 삭제: 로스(ROS), (Co. Ltd.) 등
    name = re.sub(r"\s*\([^가-힣()]*[A-Za-z][^가-힣()]*\)\s*$", "", name)
    return re.sub(r"\s+", " ", name).strip()

before = df["업체명"].nunique()
df["업체명"] = df["업체명"].map(clean_name)
print("업체명 고유값:", before, "->", df["업체명"].nunique())

업체명 고유값: 2278 -> 2263


## 3·5·6·7·8. 0원 삭제, 형 변환, 인코딩, 중복 제거

In [5]:
# 3. 실적금액 0인 행 삭제
n0 = int((df["실적금액"] == 0).sum())
df = df[df["실적금액"] != 0]
print("0원 행 삭제:", n0)

# 5. 실적일자 -> datetime
df["실적일자"] = pd.to_datetime(df["실적일자"], errors="coerce")

# 6. 벤처업체번호: 'CN' 제거 후 int
df["벤처업체번호"] = df["벤처업체번호"].astype(str).str.replace("CN", "", regex=False).astype("int64")

# 7. 수요기관코드, 품명 Label Encoding (구분용 코드이며 크기에 의미 없음)
for c in ["수요기관코드", "품명"]:
    df[c] = df[c].astype("category").cat.codes

# 8. 중복 행 제거
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print("중복 제거:", before - len(df), "행")

0원 행 삭제: 372
중복 제거: 0 행


## 저장

In [6]:
df.to_csv(DST, index=False, encoding="utf-8-sig")
print("완료:", df.shape, "->", DST)

완료: (52210, 23) -> ../data/조달청_벤처나라_전처리결과.csv
